In [1]:
import pandas as pd

In [2]:

def convert_to_pandas(dataset, batch_size=1000):
    """
    Convert a Hugging Face dataset (streaming or not) to a pandas DataFrame in batches.
    """
    df_list = []
    
    i=0
    # Hugging Face streaming datasets use iter() for batching
    for batch in dataset.iter(batch_size=batch_size):
        print(f"Processing batch {i} \n")
        i += 1
        # batch is a dict, convert directly to DataFrame
        df_batch = pd.DataFrame(batch)
        df_list.append(df_batch)
    
    df = pd.concat(df_list, ignore_index=True)
    return df

def sample_hf_data(data, sample_size=1000):

    # Shuffle the dataset (buffer_size controls memory usage)
    shuffled = data.shuffle(buffer_size=sample_size)
    # Take random samples
    sampled_dataset = shuffled.take(sample_size)
    
    return sampled_dataset

In [3]:
"""
Comprehensive Evaluation Dataset Collection from Real HuggingFace Datasets

Focuses on collecting actual datasets (not generated prompts) across diverse domains
that align with competition's emphasis on non-verifiable tasks.

Usage: python collect_comprehensive_eval.py --samples_per_domain 100
"""

from datasets import load_dataset
import json
from typing import List, Dict
import random
import argparse

random.seed(42)

def sample_hf_streaming(dataset_name: str, config: str, split: str, n_samples: int, buffer_size: int = 10000):
    """Sample from HuggingFace dataset using streaming"""
    print(f"    Streaming {n_samples} samples...", end=" ", flush=True)
    dataset = load_dataset(dataset_name, config, split=split, streaming=True)
    shuffled = dataset.shuffle(buffer_size=buffer_size, seed=42)
    
    samples = []
    for i, item in enumerate(shuffled):
        samples.append(item)
        if (i + 1) % 50 == 0:
            print(f"{i+1}", end="...", flush=True)
        if len(samples) >= n_samples:
            break
    
    print(f" Done!")
    return samples

def collect_comprehensive_datasets(samples_per_domain: int = 100):
    """
    Collect evaluation datasets from real HuggingFace datasets across diverse domains
    
    Args:
        samples_per_domain: Number of samples to collect per dataset (default: 100)
    """
    
    all_samples = []
    
    print("="*80)
    print(f"COMPREHENSIVE EVALUATION DATASET COLLECTION ({samples_per_domain} samples/domain)")
    print("="*80)
    print("\nCollecting from REAL datasets across diverse non-verifiable domains\n")
    
    domain_count = 0
    
    # ==========================================
    # SUMMARIZATION DATASETS
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] SUMMARIZATION")
    print("-" * 80)
    
    # CNN/DailyMail
    try:
        items = sample_hf_streaming("abisee/cnn_dailymail", "3.0.0", "test", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "summarization",
                "subdomain": "news_summary",
                "question": f"Summarize the following article:\n\n{item['article']}",
                "answer": item["highlights"],
                "source": "cnn_dailymail"
            })
        print(f"  ✓ CNN/DailyMail: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ CNN/DailyMail failed: {e}")
    
    # XSum (if available)
    try:
        items = sample_hf_streaming("EdinburghNLP/xsum", None, "test", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "summarization",
                "subdomain": "extreme_summary",
                "question": f"Write a one-sentence summary of:\n\n{item['document']}",
                "answer": item["summary"],
                "source": "xsum"
            })
        print(f"  ✓ XSum: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ XSum failed: {e}")
    
    # SAMSum (dialogue summarization)
    try:
        items = sample_hf_streaming("Samsung/samsum", None, "test", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "summarization",
                "subdomain": "dialogue_summary",
                "question": f"Summarize this conversation:\n\n{item['dialogue']}",
                "answer": item["summary"],
                "source": "samsum"
            })
        print(f"  ✓ SAMSum: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ SAMSum failed: {e}")
    
    # Multi-News (multi-document)
    try:
        items = sample_hf_streaming("alexfabbri/multi_news", None, "test", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "summarization",
                "subdomain": "multi_document",
                "question": f"Synthesize these multiple sources into one summary:\n\n{item['document']}",
                "answer": item["summary"],
                "source": "multi_news"
            })
        print(f"  ✓ Multi-News: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ Multi-News failed: {e}")
    
    # ==========================================
    # QUESTION ANSWERING (NON-EXTRACTIVE)
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] QUESTION ANSWERING")
    print("-" * 80)
    
    # TriviaQA
    try:
        items = sample_hf_streaming("mandarjoshi/trivia_qa", "rc.nocontext", "validation", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "question_answering",
                "subdomain": "factual_qa",
                "question": item["question"],
                "answer": item["answer"]["value"],
                "source": "triviaqa"
            })
        print(f"  ✓ TriviaQA: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ TriviaQA failed: {e}")
    
    # ELI5 (explain like I'm 5)
    try:
        items = sample_hf_streaming("eli5_category", None, "test", samples_per_domain, buffer_size=5000)
        for item in items:
            all_samples.append({
                "domain": "question_answering",
                "subdomain": "explanatory_qa",
                "question": item["title"],
                "answer": item["answers"]["text"][0] if item["answers"]["text"] else "",
                "source": "eli5"
            })
        print(f"  ✓ ELI5: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ ELI5 failed: {e}")
    
    # MS MARCO (passage ranking/QA)
    try:
        items = sample_hf_streaming("microsoft/ms_marco", "v2.1", "validation", samples_per_domain)
        for item in items:
            if item.get("passages") and item["passages"].get("passage_text"):
                all_samples.append({
                    "domain": "question_answering",
                    "subdomain": "passage_qa",
                    "question": item["query"],
                    "answer": item["passages"]["passage_text"][0],
                    "source": "ms_marco"
                })
        print(f"  ✓ MS MARCO: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ MS MARCO failed: {e}")
    
    # ==========================================
    # READING COMPREHENSION
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] READING COMPREHENSION")
    print("-" * 80)
    
    # DROP
    try:
        items = sample_hf_streaming("ucinlp/drop", None, "validation", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "reading_comprehension",
                "subdomain": "discrete_reasoning",
                "question": f"Context: {item['passage']}\n\nQuestion: {item['question']}",
                "answer": item["answers_spans"]["spans"][0] if item["answers_spans"]["spans"] else str(item["answers_spans"]["number"]),
                "source": "drop"
            })
        print(f"  ✓ DROP: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ DROP failed: {e}")
    
    # QuAC (Question Answering in Context - conversational)
    try:
        items = sample_hf_streaming("quac", None, "validation", samples_per_domain)
        for item in items:
            # Get first question-answer pair
            if item["questions"] and item["answers"]:
                all_samples.append({
                    "domain": "reading_comprehension",
                    "subdomain": "conversational_qa",
                    "question": f"Context: {item['context']}\n\nQuestion: {item['questions'][0]}",
                    "answer": item["answers"]["texts"][0][0] if item["answers"]["texts"] else "",
                    "source": "quac"
                })
        print(f"  ✓ QuAC: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ QuAC failed: {e}")
    
    # BoolQ (yes/no questions)
    try:
        items = sample_hf_streaming("google/boolq", None, "validation", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "reading_comprehension",
                "subdomain": "boolean_qa",
                "question": f"Passage: {item['passage']}\n\nQuestion: {item['question']}",
                "answer": "Yes" if item["answer"] else "No",
                "source": "boolq"
            })
        print(f"  ✓ BoolQ: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ BoolQ failed: {e}")
    
    # ==========================================
    # COMMONSENSE REASONING
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] COMMONSENSE REASONING")
    print("-" * 80)
    
    # CommonsenseQA
    try:
        csqa = load_dataset("tau/commonsense_qa", split="validation")
        print(f"    Loading CommonsenseQA...", end=" ", flush=True)
        items = list(csqa)[:samples_per_domain]
        for item in items:
            question = item['question']
            choices = item['choices']
            question += "\n" + "\n".join([f"{choices['label'][i]}) {choices['text'][i]}" 
                                         for i in range(len(choices['label']))])
            all_samples.append({
                "domain": "commonsense_reasoning",
                "subdomain": "commonsense_qa",
                "question": question,
                "answer": item['answerKey'],
                "source": "commonsense_qa"
            })
        print(f"Done! {samples_per_domain} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # PIQA (Physical Commonsense)
    try:
        piqa = load_dataset("ybisk/piqa", split="validation")
        print(f"    Loading PIQA...", end=" ", flush=True)
        items = list(piqa)[:samples_per_domain]
        for item in items:
            question = f"{item['goal']}\nA) {item['sol1']}\nB) {item['sol2']}"
            all_samples.append({
                "domain": "commonsense_reasoning",
                "subdomain": "physical_commonsense",
                "question": question,
                "answer": "A" if item["label"] == 0 else "B",
                "source": "piqa"
            })
        print(f"Done! {samples_per_domain} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # Social IQA
    try:
        items = sample_hf_streaming("social_i_qa", None, "validation", samples_per_domain)
        for item in items:
            question = f"{item['context']}\n\nQuestion: {item['question']}\nA) {item['answerA']}\nB) {item['answerB']}\nC) {item['answerC']}"
            answer_map = {"1": "A", "2": "B", "3": "C"}
            all_samples.append({
                "domain": "commonsense_reasoning",
                "subdomain": "social_reasoning",
                "question": question,
                "answer": answer_map.get(item["label"], "A"),
                "source": "social_iqa"
            })
        print(f"  ✓ Social IQA: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ Social IQA failed: {e}")
    
    # Winogrande (commonsense reasoning via pronoun resolution)
    try:
        wino = load_dataset("allenai/winogrande", "winogrande_xl", split="validation")
        print(f"    Loading Winogrande...", end=" ", flush=True)
        items = list(wino)[:samples_per_domain]
        for item in items:
            question = f"{item['sentence']}\nA) {item['option1']}\nB) {item['option2']}"
            all_samples.append({
                "domain": "commonsense_reasoning",
                "subdomain": "pronoun_resolution",
                "question": question,
                "answer": "A" if item["answer"] == "1" else "B",
                "source": "winogrande"
            })
        print(f"Done! {samples_per_domain} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # ==========================================
    # SCIENCE REASONING
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] SCIENCE REASONING")
    print("-" * 80)
    
    # ARC Challenge
    try:
        arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
        print(f"    Loading ARC Challenge...", end=" ", flush=True)
        items = list(arc)[:samples_per_domain]
        for item in items:
            question = item['question']
            choices = item['choices']
            question += "\n" + "\n".join([f"{choices['label'][i]}) {choices['text'][i]}" 
                                         for i in range(len(choices['label']))])
            all_samples.append({
                "domain": "science_reasoning",
                "subdomain": "science_qa",
                "question": question,
                "answer": item['answerKey'],
                "source": "arc_challenge"
            })
        print(f"Done! {samples_per_domain} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # SciQ (science questions)
    try:
        items = sample_hf_streaming("allenai/sciq", None, "test", samples_per_domain)
        for item in items:
            question = f"{item['question']}\nA) {item['distractor1']}\nB) {item['distractor2']}\nC) {item['distractor3']}\nD) {item['correct_answer']}"
            all_samples.append({
                "domain": "science_reasoning",
                "subdomain": "science_mcq",
                "question": question,
                "answer": "D",  # Correct answer always D in this format
                "source": "sciq"
            })
        print(f"  ✓ SciQ: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ SciQ failed: {e}")
    
    # ==========================================
    # NATURAL LANGUAGE INFERENCE
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] NATURAL LANGUAGE INFERENCE")
    print("-" * 80)
    
    # SNLI
    try:
        items = sample_hf_streaming("stanfordnlp/snli", None, "test", samples_per_domain)
        label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
        for item in items:
            if item["label"] != -1:  # Skip unlabeled
                all_samples.append({
                    "domain": "nli",
                    "subdomain": "textual_entailment",
                    "question": f"Premise: {item['premise']}\nHypothesis: {item['hypothesis']}\n\nWhat is the relationship?",
                    "answer": label_map[item["label"]],
                    "source": "snli"
                })
        print(f"  ✓ SNLI: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ SNLI failed: {e}")
    
    # ANLI (Adversarial NLI)
    try:
        items = sample_hf_streaming("facebook/anli", None, "test_r1", samples_per_domain//3)
        label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
        for item in items:
            all_samples.append({
                "domain": "nli",
                "subdomain": "adversarial_nli",
                "question": f"Premise: {item['premise']}\nHypothesis: {item['hypothesis']}\n\nRelationship?",
                "answer": label_map[item["label"]],
                "source": "anli"
            })
        print(f"  ✓ ANLI: {samples_per_domain//3} samples")
    except Exception as e:
        print(f"  ✗ ANLI failed: {e}")
    
    # ==========================================
    # PARAPHRASE & SIMILARITY
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] PARAPHRASE & SIMILARITY")
    print("-" * 80)
    
    # PAWS (paraphrase adversaries)
    try:
        items = sample_hf_streaming("google-research-datasets/paws", "labeled_final", "test", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "paraphrase",
                "subdomain": "paraphrase_detection",
                "question": f"Sentence 1: {item['sentence1']}\nSentence 2: {item['sentence2']}\n\nAre these paraphrases?",
                "answer": "Yes" if item["label"] == 1 else "No",
                "source": "paws"
            })
        print(f"  ✓ PAWS: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ PAWS failed: {e}")
    
    # QQP (Quora Question Pairs)
    try:
        items = sample_hf_streaming("quora", None, "train", samples_per_domain, buffer_size=50000)
        for item in items:
            all_samples.append({
                "domain": "paraphrase",
                "subdomain": "question_similarity",
                "question": f"Question 1: {item['questions']['text'][0]}\nQuestion 2: {item['questions']['text'][1]}\n\nAre these duplicate questions?",
                "answer": "Yes" if item["is_duplicate"] else "No",
                "source": "qqp"
            })
        print(f"  ✓ QQP: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ QQP failed: {e}")
    
    # ==========================================
    # SENTIMENT & EMOTION
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] SENTIMENT & EMOTION ANALYSIS")
    print("-" * 80)
    
    # SST-2 (sentiment)
    try:
        items = sample_hf_streaming("stanfordnlp/sst2", None, "validation", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "sentiment",
                "subdomain": "binary_sentiment",
                "question": f"What is the sentiment of this text: \"{item['sentence']}\"",
                "answer": "positive" if item["label"] == 1 else "negative",
                "source": "sst2"
            })
        print(f"  ✓ SST-2: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ SST-2 failed: {e}")
    
    # Emotion (multi-class emotion)
    try:
        items = sample_hf_streaming("dair-ai/emotion", None, "test", samples_per_domain)
        emotion_map = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
        for item in items:
            all_samples.append({
                "domain": "sentiment",
                "subdomain": "emotion_classification",
                "question": f"What emotion does this express: \"{item['text']}\"",
                "answer": emotion_map[item["label"]],
                "source": "emotion"
            })
        print(f"  ✓ Emotion: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ Emotion failed: {e}")
    
    # ==========================================
    # DIALOGUE & CONVERSATION
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] DIALOGUE & CONVERSATION")
    print("-" * 80)
    
    # DailyDialog
    try:
        items = sample_hf_streaming("daily_dialog", None, "test", samples_per_domain)
        for item in items:
            if item["dialog"]:
                # Take first few turns as context, predict next
                context = "\n".join(item["dialog"][:3])
                all_samples.append({
                    "domain": "dialogue",
                    "subdomain": "dialogue_generation",
                    "question": f"Continue this conversation:\n{context}",
                    "answer": item["dialog"][3] if len(item["dialog"]) > 3 else "",
                    "source": "daily_dialog"
                })
        print(f"  ✓ DailyDialog: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ DailyDialog failed: {e}")
    
    # Empathetic Dialogues
    try:
        items = sample_hf_streaming("empathetic_dialogues", None, "test", samples_per_domain)
        for item in items:
            all_samples.append({
                "domain": "dialogue",
                "subdomain": "empathetic_response",
                "question": f"Context: {item['context']}\nPrompt: {item['prompt']}\n\nRespond empathetically:",
                "answer": item["utterance"],
                "source": "empathetic_dialogues"
            })
        print(f"  ✓ Empathetic Dialogues: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ Empathetic Dialogues failed: {e}")
    
    # ==========================================
    # WRITING & REWRITING
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] WRITING & REWRITING")
    print("-" * 80)
    
    # Parabank (paraphrase generation)
    try:
        items = sample_hf_streaming("parabank/parabank", None, "train", samples_per_domain, buffer_size=50000)
        for item in items:
            if "source" in item and "target" in item:
                all_samples.append({
                    "domain": "rewriting",
                    "subdomain": "paraphrase_generation",
                    "question": f"Paraphrase this sentence: {item['source']}",
                    "answer": item["target"],
                    "source": "parabank"
                })
        print(f"  ✓ Parabank: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ Parabank failed: {e}")
    
    # JFLEG (grammar correction)
    try:
        items = sample_hf_streaming("jfleg", None, "test", samples_per_domain)
        for item in items:
            if item["corrections"]:
                all_samples.append({
                    "domain": "rewriting",
                    "subdomain": "grammar_correction",
                    "question": f"Correct the grammar: {item['sentence']}",
                    "answer": item["corrections"][0],
                    "source": "jfleg"
                })
        print(f"  ✓ JFLEG: {samples_per_domain} samples")
    except Exception as e:
        print(f"  ✗ JFLEG failed: {e}")
    
    # ==========================================
    # KNOWLEDGE & FACTS
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] KNOWLEDGE & FACTS")
    print("-" * 80)
    
    # MMLU (if space available)
    try:
        mmlu = load_dataset("cais/mmlu", "all", split="test")
        print(f"    Loading MMLU...", end=" ", flush=True)
        items = list(mmlu)[:samples_per_domain]
        for item in items:
            question = item['question']
            choices = item['choices']
            question += "\n" + "\n".join([f"{chr(65+i)}) {choices[i]}" for i in range(len(choices))])
            all_samples.append({
                "domain": "knowledge",
                "subdomain": "multitask_knowledge",
                "question": question,
                "answer": chr(65 + item['answer']),
                "source": "mmlu"
            })
        print(f"Done! {samples_per_domain} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # ==========================================
    # MATH (LOW PRIORITY - Small samples)
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] MATH (Low Priority - {samples_per_domain//2} samples)")
    print("-" * 80)
    
    # GSM8K
    try:
        gsm8k = load_dataset("openai/gsm8k", "main", split="test")
        print(f"    Loading GSM8K...", end=" ", flush=True)
        items = list(gsm8k)[:samples_per_domain//2]
        for item in items:
            answer = item["answer"]
            if "####" in answer:
                answer = answer.split("####")[1].strip()
            all_samples.append({
                "domain": "math",
                "subdomain": "grade_school_math",
                "question": item["question"],
                "answer": answer,
                "source": "gsm8k"
            })
        print(f"Done! {samples_per_domain//2} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # ==========================================
    # CODE (LOW PRIORITY - Small samples)
    # ==========================================
    domain_count += 1
    print(f"\n[{domain_count}] CODE (Low Priority - {samples_per_domain//2} samples)")
    print("-" * 80)
    
    # HumanEval
    try:
        humaneval = load_dataset("openai/openai_humaneval", split="test")
        print(f"    Loading HumanEval...", end=" ", flush=True)
        items = list(humaneval)[:samples_per_domain//2]
        for item in items:
            all_samples.append({
                "domain": "code",
                "subdomain": "code_generation",
                "question": item["prompt"],
                "answer": item["canonical_solution"],
                "source": "humaneval"
            })
        print(f"Done! {samples_per_domain//2} samples")
    except Exception as e:
        print(f"Failed: {e}")
    
    # ==========================================
    # SUMMARY
    # ==========================================
    print("\n" + "="*80)
    print("COLLECTION COMPLETE")
    print("="*80)
    print(f"\nTotal samples: {len(all_samples)}")
    
    # Count by domain
    domain_counts = {}
    subdomain_counts = {}
    for sample in all_samples:
        domain = sample['domain']
        subdomain = sample.get('subdomain', 'other')
        domain_counts[domain] = domain_counts.get(domain, 0) + 1
        subdomain_counts[subdomain] = subdomain_counts.get(subdomain, 0) + 1
    
    print("\nDomain distribution:")
    for domain, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
        print(f"  {domain:25s}: {count:4d} samples")
    
    print(f"\nTotal unique domains: {len(domain_counts)}")
    print(f"Total unique subdomains: {len(subdomain_counts)}")
    
    return all_samples

    


d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# parser = argparse.ArgumentParser(description='Collect comprehensive evaluation datasets')
# parser.add_argument('--samples_per_domain', type=int, default=200,
#                     help='Number of samples to collect per domain (default: 100)')

# args = parser.parse_args()

eval_data = collect_comprehensive_datasets(samples_per_domain=200)


COMPREHENSIVE EVALUATION DATASET COLLECTION (200 samples/domain)



[1] SUMMARIZATION
--------------------------------------------------------------------------------
    Streaming 200 samples... 50...100...150...200... Done!
  ✓ CNN/DailyMail: 200 samples
    Streaming 200 samples...   ✗ XSum failed: Dataset scripts are no longer supported, but found xsum.py
    Streaming 200 samples...   ✗ SAMSum failed: Dataset 'Samsung/samsum' doesn't exist on the Hub or cannot be accessed.
    Streaming 200 samples...   ✗ Multi-News failed: Dataset scripts are no longer supported, but found multi_news.py

[2] QUESTION ANSWERING
--------------------------------------------------------------------------------
    Streaming 200 samples... 50...100...150...200... Done!
  ✓ TriviaQA: 200 samples
    Streaming 200 samples...   ✗ ELI5 failed: Dataset scripts are no longer supported, but found eli5_category.py
    Streaming 200 samples... 50...100...150...200... Done!
  ✓ MS MARCO: 200 samples

[3] READIN

In [5]:
eval_df=pd.DataFrame(eval_data)
eval_df

,domain,subdomain,question,answer,source
0,summarization,news_summary,Summarize the following article:\n\n(CNN)Nobel...,Grass tried in his literature to come to grips...,cnn_dailymail
1,summarization,news_summary,Summarize the following article:\n\nCosmetic s...,"Rhiannon Langley, from Melbourne, is undergoin...",cnn_dailymail
2,summarization,news_summary,Summarize the following article:\n\nA teenager...,Teen with deadly brain tumour was told by doct...,cnn_dailymail
3,summarization,news_summary,Summarize the following article:\n\nThe first ...,New Jersey Institute of Technology research re...,cnn_dailymail
4,summarization,news_summary,Summarize the following article:\n\nMotherwell...,Motherwell will offer free entry to their regu...,cnn_dailymail
...,...,...,...,...,...
3255,code,code_generation,"\ndef check_dict_case(dict):\n """"""\n Giv...",if len(dict.keys()) == 0:\n return ...,humaneval
3256,code,code_generation,"\ndef count_up_to(n):\n """"""Implement a func...","primes = []\n for i in range(2, n):\n ...",humaneval
3257,code,code_generation,"\ndef multiply(a, b):\n """"""Complete the fun...",return abs(a % 10) * abs(b % 10)\n,humaneval
3258,code,code_generation,"\ndef count_upper(s):\n """"""\n Given a st...","count = 0\n for i in range(0,len(s),2):...",humaneval


In [8]:
eval_df.to_parquet('./eval_data/benchmark_oos.parquet')

In [6]:
eval_df.groupby(['domain']).size()

domain
code                     100
commonsense_reasoning    400
knowledge                200
math                     100
nli                      260
paraphrase               200
question_answering       400
reading_comprehension    400
rewriting                200
science_reasoning        400
sentiment                400
summarization            200
dtype: int64